<a href="https://colab.research.google.com/github/peanutpirate/Week_7_CodeCamp_2026_Projects_And_Learning_Materials/blob/Projects_Daily/16_tensorflow_car_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Car Price Prediction System (TensorFlow)

## Proje Amacı

Bu projede araç özelliklerine bakarak **araba fiyatını tahmin eden bir makine öğrenmesi modeli** geliştirilmektedir.

Model şu bilgileri kullanır:

- Model yılı
- Kilometre
- Motor hacmi
- Beygir gücü
- Araç yaşı

Hedef değişken:

**Araç fiyatı (Price)**

Bu nedenle problem bir **regresyon problemidir.**

---

## Kullanılan TensorFlow Konuları

Bu projede şu başlıklar uygulanmıştır:

- Veri hazırlama
- Train / Test ayrımı
- TensorFlow Sequential modeli
- Dense katmanlar
- Model eğitimi (training)
- Model değerlendirmesi
- Yeni veri ile tahmin

Amaç:

**Uçtan uca bir makine öğrenmesi pipeline'ı kurmak.**

In [ ]:
### 🟦 Hücre 2 – Veri Hazırlama (NumPy / Pandas)

import numpy as np
import pandas as pd
import tensorflow as tf

np.random.seed(42)

# Sentetik veri üretimi

n = 500

model_year = np.random.randint(2005, 2024, n)
km = np.random.randint(5000, 250000, n)
engine_size = np.random.uniform(1.0, 4.0, n)
horsepower = np.random.randint(70, 400, n)
age = 2024 - model_year

# Fiyat formülü (simülasyon)

price = (
    model_year * 1200
    - km * 0.05
    + engine_size * 15000
    + horsepower * 200
    - age * 3000
    + np.random.normal(0, 5000, n)
)

data = pd.DataFrame({
    "model_year": model_year,
    "km": km,
    "engine_size": engine_size,
    "horsepower": horsepower,
    "age": age,
    "price": price
})

print(data.head())

# Feature ve target

X = data.drop("price", axis=1)
y = data["price"]

# Basit normalizasyon

X = (X - X.mean()) / X.std()

print("Feature shape:", X.shape)

   model_year      km  engine_size  horsepower  age         price
0        2011  171981     3.237924         244   13  2.468125e+06
1        2019  225411     2.225555         153    5  2.461055e+06
2        2015   24870     3.798814          91    9  2.467252e+06
3        2012  170650     3.972788          96   12  2.459249e+06
4        2011  208196     1.615005         264   13  2.434952e+06
Feature shape: (500, 5)


In [ ]:
### 🟦 Hücre 3 – Train / Test Bölmesi
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (400, 5)
Test size: (100, 5)


In [ ]:
### 🟦 Hücre 4 – Model Oluşturma
model = tf.keras.Sequential([

    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),

    tf.keras.layers.Dense(32, activation='relu'),

    tf.keras.layers.Dense(1)  # regresyon -> aktivasyon yok
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
### 🟦 Hücre 5 – Model Derleme (Compile)

model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,497 (9.75 KB)

 Trainable params: 2,497 (9.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
### 🟦 Hücre 6 – Training
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 6068497809408.0000 - mae: 2463199.7500 - val_loss: 6094844854272.0000 - val_mae: 2468508.7500
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 6069664350208.0000 - mae: 2463422.0000 - val_loss: 6094841708544.0000 - val_mae: 2468508.2500
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 6069564211200.0000 - mae: 2463409.0000 - val_loss: 6094838562816.0000 - val_mae: 2468507.2500
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 6082119860224.0000 - mae: 2465936.7500 - val_loss: 6094834368512.0000 - val_mae: 2468506.7500
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6071432249344.0000 - mae: 2463784.5000 - val_loss: 6094829649920.0000 - val_mae: 2468505.7500
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 6073934675968.0000 - mae: 2464277.5000 - val_loss: 6094821785600.0000 - val_mae: 2468504.5000
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 6066099716096.0000 - mae

In [ ]:
### 🟦 Hücre 7 – Model Değerlendirmesi -ÖNEMLİ
loss, mae = model.evaluate(X_test, y_test)

print("Test MSE:", loss)
print("Test MAE:", mae)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 6058594009088.0000 - mae: 2461135.2500 
Test MSE: 6053432918016.0
Test MAE: 2460102.75


In [ ]:
### 🟦 Hücre 8 – Model Tahminleri
# Yeni araç verisi

new_car = pd.DataFrame({
    "model_year": [2020],
    "km": [45000],
    "engine_size": [1.6],
    "horsepower": [130],
    "age": [4]
})

# aynı normalizasyon

new_car = (new_car - X.mean()) / X.std()

prediction = model.predict(new_car)

print("Tahmini fiyat:", prediction[0][0], "TL")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Tahmini fiyat: 20149212.0 TL


### 🟦 Hücre 9 – Araba Fiyatları Analizi (Yorum)

Model sonuçlarına göre bazı değişkenler fiyat üzerinde daha güçlü etkiye sahip olabilir.

Özellikle:

- Motor hacmi
- Beygir gücü
- Araç yaşı

fiyatı önemli ölçüde etkileyen faktörlerdir.

Kilometre ise genellikle fiyatı düşüren bir faktördür.

Model özellikle:

- orta segment araçlar
- standart kullanım kilometresi olan araçlar

için daha doğru tahminler yapabilir.

Ancak lüks araçlar veya çok nadir modeller için daha fazla veri gerekebilir.

### 🟦 Son Hücre – Yönetici Özeti
# Yönetici Özeti

• Model araç fiyatlarını tahmin etmek için eğitildi.

• Ortalama tahmin hatası yaklaşık olarak model performansına bağlıdır.

• Sistem ikinci el araç fiyatlandırmasında kullanılabilir.

• Daha fazla veri ve gerçek piyasa verisi ile model performansı önemli ölçüde artırılabilir.

Bu sistem otomotiv şirketleri için:

- fiyat öneri sistemi
- otomatik değerleme
- ikinci el pazar analizi

amacıyla kullanılabilir.
